# 04 · SimGCD baseline (DINO ViT-B/16) on the v2 benchmark

An end-to-end generalized category discovery (GCD) baseline that fine-tunes the backbone. This notebook trains **SimGCD**
(Wen et al., ICCV 2023) on the v2 iNaturalist benchmark with the official losses and hyperparameters, and adds
two practical additions: **resume after any interruption** and **DataLoader workers on Windows**.

| | |
|---|---|
| Old (labelled) classes | the 1,177 `known` species; 50% of their images carry labels (SimGCD default) |
| New classes | 180 `novel_known_branch` + 460 `novel_unseen_branch` = 640 species |
| Excluded | 200 `pool_only` species (not part of the benchmark) |
| Model | DINO ViT-B/16, last block fine-tuned, `DINOHead` with one prototype per species (oracle k, as in SimGCD) |
| Reported | ACC All/Old/New on the unlabelled images (SimGCD protocol), plus NMI / ARI / pair-F1 with the all-singletons baseline on each novel role |

**Imported unchanged from the official repo (`files\SimGCD\model.py`):** `DINOHead`, `SupConLoss`, `info_nce_logits`,
`DistillLoss`, `get_params_groups`. The training step is copied line for line from the repo's `train.py`.
The transforms are identical to the repo's `get_transform('imagenet')`.

**Model selection:** none. The final epoch is reported, because choosing a "best" epoch would use the labels
of the novel species.

**Reads** `files\v2\species_split_v2.json`, `files\v2\items.json` and the images. **Writes only** to `files\v2\simgcd_full\`.

**Order:** run Steps 1–7, then the smoke test (Step 8) and check its projected time. Change `EPOCHS` in Step 1
*before* starting Step 9 if needed. If the run stops for any reason, rerun Steps 1–7 and then Step 9: it resumes
from the last finished epoch.

## Step 1 — Configuration

In [ ]:
# CELL: config
import json, os, sys, time, math, random, inspect
from pathlib import Path
from collections import Counter
import numpy as np
import torch
import torch.nn as nn

BASE_DIR   = Path(os.environ.get("HNSD_BASE_DIR", r"C:\hnsd\files"))   # data root; set HNSD_BASE_DIR or edit
IMG_ROOT   = BASE_DIR / "selected"              # images: selected/train_mini/<species folder>/*.jpg
V2_DIR     = BASE_DIR / "v2"                    # read only
SIMGCD_DIR = BASE_DIR / "SimGCD"                # official repo, read only
OUT_DIR    = V2_DIR / "simgcd_full"             # everything this notebook writes

SEED = 42
EPOCHS = 100            # fixed once Step 9 starts: the LR and teacher-temperature schedules depend on it
                        # (paper: 200; Step 8 prints the projected time so you can choose)
BATCH_SIZE  = 128
NUM_WORKERS = 6         # set to 0 if Step 8 reports a worker error
EVAL_EVERY  = 10        # epochs between evaluations on the unlabelled images (monitoring only)
PRINT_EVERY = 100       # iterations between progress lines

# SimGCD hyperparameters, as in the repo's scripts/run_cub.sh (balanced fine-grained setting)
LR, MOMENTUM, WEIGHT_DECAY = 0.1, 0.9, 5e-5
GRAD_FROM_BLOCK = 11
SUP_WEIGHT      = 0.35
MEMAX_WEIGHT    = 2.0   # 2 for balanced classes (CUB); 1 is used for long-tailed Herbarium19 and Aircraft
WARMUP_TEACHER_TEMP, TEACHER_TEMP, WARMUP_TEACHER_TEMP_EPOCHS = 0.07, 0.04, 30
N_VIEWS, PROP_TRAIN_LABELS = 2, 0.5
IMAGE_SIZE, CROP_PCT = 224, 0.875
FP16 = True
DINO_PRETRAINED = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR.mkdir(parents=True, exist_ok=True)
assert WARMUP_TEACHER_TEMP_EPOCHS <= EPOCHS
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
print(f"device {DEVICE}" + (f" ({torch.cuda.get_device_name(0)}, "
      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)" if DEVICE == "cuda" else ""))
print("torch", torch.__version__)

## Step 2 — Import the official SimGCD components

The signatures are printed so they can be checked against the repo.

In [ ]:
# CELL: simgcd_imports
import torchvision, scipy, sklearn
if str(SIMGCD_DIR) not in sys.path:
    sys.path.insert(0, str(SIMGCD_DIR))
from model import DINOHead, info_nce_logits, SupConLoss, DistillLoss, get_params_groups

for f in (DINOHead.__init__, info_nce_logits, SupConLoss.__init__, DistillLoss.__init__):
    print(f.__qualname__, inspect.signature(f))
print("torchvision", torchvision.__version__, "| scipy", scipy.__version__, "| scikit-learn", sklearn.__version__)

## Step 3 — Class split and labelled subset

Old classes get indices `0 … n_old-1` and new classes follow, as in SimGCD. Within each old species, half the
images (seeded) are labelled. The split is saved on the first run and **reloaded on every later run**, so a resumed
run always uses exactly the same labels.

In [ ]:
# CELL: split
S = json.load(open(V2_DIR / "species_split_v2.json"))
ITEMS = json.load(open(V2_DIR / "items.json"))                 # [relpath, category_id]
ROLE = {s["category_id"]: s["role"] for s in S}

old_sp = sorted(c for c, r in ROLE.items() if r == "known")
new_sp = sorted(c for c, r in ROLE.items() if r in ("novel_known_branch", "novel_unseen_branch"))
CLASS_OF = {c: i for i, c in enumerate(old_sp + new_sp)}
N_OLD, N_CLASSES = len(old_sp), len(old_sp) + len(new_sp)

split_path = OUT_DIR / "simgcd_split.json"
if split_path.exists():
    SPLIT = json.load(open(split_path))
    assert SPLIT["old_species"] == old_sp and SPLIT["new_species"] == new_sp, "split file does not match the benchmark"
    print("loaded the saved split")
else:
    rng = np.random.default_rng(SEED)
    by_sp = {}
    for rel, c in ITEMS:
        if c in CLASS_OF:
            by_sp.setdefault(c, []).append(rel)
    labelled, unlabelled = [], []
    for c in old_sp + new_sp:
        rels = sorted(by_sp[c])
        if c in old_sp:
            perm = rng.permutation(len(rels)); k = int(round(PROP_TRAIN_LABELS * len(rels)))
            labelled += [[rels[i], c] for i in sorted(perm[:k])]
            unlabelled += [[rels[i], c] for i in sorted(perm[k:])]
        else:
            unlabelled += [[r, c] for r in rels]
    SPLIT = {"seed": SEED, "prop_train_labels": PROP_TRAIN_LABELS, "old_species": old_sp, "new_species": new_sp,
             "labelled": labelled, "unlabelled": unlabelled}
    json.dump(SPLIT, open(split_path, "w"))
    print("saved", split_path)

LAB, UNL = SPLIT["labelled"], SPLIT["unlabelled"]
assert not ({c for _, c in LAB} & set(new_sp)), "LEAKAGE: a new species has labelled images"
missing = [r for r, _ in LAB + UNL if not (IMG_ROOT / r).exists()]
assert not missing, f"{len(missing)} images missing, e.g. {missing[:3]}"
roles_unl = Counter(ROLE[c] for _, c in UNL)
print(f"classes: {N_OLD} old + {N_CLASSES - N_OLD} new = {N_CLASSES}")
print(f"labelled images {len(LAB)} | unlabelled images {len(UNL)} {dict(roles_unl)}")
print(f"excluded pool_only species: {sum(r == 'pool_only' for r in ROLE.values())}")

## Step 4 — Datasets in an importable module

On Windows, Jupyter cannot send classes defined inside the notebook to DataLoader worker processes. Writing the
dataset to a small `.py` file and importing it removes that limit, so `NUM_WORKERS > 0` works.

In [ ]:
# CELL: data_module
DATA_MODULE = r"""
from pathlib import Path
import numpy as np
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
from torchvision.transforms import InterpolationMode

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

def get_transforms(image_size=224, crop_pct=0.875):
    # identical to SimGCD data/augmentations get_transform('imagenet') with interpolation=3 (bicubic)
    resize = transforms.Resize(int(image_size / crop_pct), InterpolationMode.BICUBIC)
    train = transforms.Compose([resize, transforms.RandomCrop(image_size), transforms.RandomHorizontalFlip(p=0.5),
                                transforms.ColorJitter(), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
    test = transforms.Compose([resize, transforms.CenterCrop(image_size), transforms.ToTensor(),
                               transforms.Normalize(MEAN, STD)])
    return train, test

class Views:
    # same as SimGCD ContrastiveLearningViewGenerator
    def __init__(self, base, n_views=2):
        self.base, self.n_views = base, n_views
    def __call__(self, x):
        return [self.base(x) for _ in range(self.n_views)]

class Items(Dataset):
    # returns (image or list of views, class index, item index, labelled flag)
    def __init__(self, root, rels, labels, transform, labelled_flags=None):
        self.root, self.rels = str(root), list(rels)
        self.labels = np.asarray(labels, dtype=np.int64)
        self.flags = np.zeros(len(self.rels), np.int64) if labelled_flags is None else np.asarray(labelled_flags, np.int64)
        self.transform = transform
    def __len__(self):
        return len(self.rels)
    def __getitem__(self, i):
        with Image.open(Path(self.root) / self.rels[i]) as im:
            img = im.convert("RGB")
        return self.transform(img), int(self.labels[i]), i, int(self.flags[i])
"""
(OUT_DIR / "hnsd_simgcd_data.py").write_text(DATA_MODULE.lstrip())
if str(OUT_DIR) not in sys.path:
    sys.path.insert(0, str(OUT_DIR))
import importlib, hnsd_simgcd_data as hd
importlib.reload(hd)

train_tf, test_tf = hd.get_transforms(IMAGE_SIZE, CROP_PCT)
TRAIN_SET = hd.Items(IMG_ROOT, [r for r, _ in LAB] + [r for r, _ in UNL],
                     [CLASS_OF[c] for _, c in LAB] + [CLASS_OF[c] for _, c in UNL],
                     hd.Views(train_tf, N_VIEWS), [1] * len(LAB) + [0] * len(UNL))      # labelled first, as in SimGCD
UNL_TEST = hd.Items(IMG_ROOT, [r for r, _ in UNL], [CLASS_OF[c] for _, c in UNL], test_tf)
UNL_CAT  = np.array([c for _, c in UNL])
UNL_ROLE = np.array([ROLE[c] for c in UNL_CAT])

# SimGCD sampler: balances labelled and unlabelled images in every batch
label_len, unlabelled_len = len(LAB), len(UNL)
SAMPLE_WEIGHTS = torch.DoubleTensor([1.0] * label_len + [label_len / unlabelled_len] * unlabelled_len)

def make_train_loader(num_workers=NUM_WORKERS):
    sampler = torch.utils.data.WeightedRandomSampler(SAMPLE_WEIGHTS, num_samples=len(TRAIN_SET),
                                                     generator=torch.Generator().manual_seed(SEED))
    kw = dict(num_workers=num_workers, pin_memory=(DEVICE == "cuda"))
    if num_workers > 0:
        kw.update(persistent_workers=True, multiprocessing_context="spawn", prefetch_factor=4)
    return torch.utils.data.DataLoader(TRAIN_SET, batch_size=BATCH_SIZE, sampler=sampler, drop_last=True, **kw), sampler

def make_test_loader(ds, num_workers=NUM_WORKERS, batch_size=256):
    kw = dict(num_workers=num_workers)
    if num_workers > 0:
        kw.update(multiprocessing_context="spawn")
    return torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=False, **kw)

x, y, i, f = TRAIN_SET[0]
print(f"train set {len(TRAIN_SET)} | views {len(x)} x {tuple(x[0].shape)} | first item class {y} labelled {f}")
print(f"iterations per epoch: {len(TRAIN_SET) // BATCH_SIZE}")

## Step 5 — Model

DINO ViT-B/16 from `torch.hub` (the cached copy from the earlier run is used), all blocks frozen except block 11,
then the SimGCD `DINOHead` with one prototype per species.

In [ ]:
# CELL: model
def build_model():
    backbone = torch.hub.load("facebookresearch/dino:main", "dino_vitb16", pretrained=DINO_PRETRAINED)
    for p in backbone.parameters():
        p.requires_grad = False
    for name, p in backbone.named_parameters():          # exactly as SimGCD train.py
        if "block" in name and int(name.split(".")[1]) >= GRAD_FROM_BLOCK:
            p.requires_grad = True
    projector = DINOHead(in_dim=backbone.embed_dim, out_dim=N_CLASSES, nlayers=3)
    return nn.Sequential(backbone, projector).to(DEVICE)

_m = build_model()
n_tr = sum(p.numel() for p in _m.parameters() if p.requires_grad)
n_all = sum(p.numel() for p in _m.parameters())
print(f"parameters: {n_all/1e6:.1f} M total, {n_tr/1e6:.1f} M trainable | head outputs {N_CLASSES}")
del _m
if DEVICE == "cuda":
    torch.cuda.empty_cache()

## Step 6 — Evaluation

- **ACC All / Old / New:** SimGCD's `split_cluster_acc_v2` (one Hungarian matching over all classes), on the
  unlabelled images.
- **Per novel role** (`novel_known_branch`, `novel_unseen_branch`): NMI, ARI and pair-F1 of the predicted
  species, each with the all-singletons partition beside it (the lesson from v2: NMI alone is saturated).

In [ ]:
# CELL: eval
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import normalized_mutual_info_score as NMI, adjusted_rand_score as ARI
from sklearn.metrics.cluster import contingency_matrix

def split_cluster_acc_v2(y_true, y_pred, mask):
    # copied from SimGCD util/cluster_and_log_utils.py (single-process path)
    y_true = y_true.astype(int)
    old_gt, new_gt = set(y_true[mask]), set(y_true[~mask])
    D = max(y_pred.max(), y_true.max()) + 1
    w = np.zeros((D, D), dtype=np.int64)
    np.add.at(w, (y_pred, y_true), 1)
    ind = np.vstack(linear_sum_assignment(w.max() - w)).T
    ind_map = {j: i for i, j in ind}
    total = sum(w[i, j] for i, j in ind) / y_pred.size
    old = sum(w[ind_map[i], i] for i in old_gt) / max(sum(w[:, i].sum() for i in old_gt), 1)
    new = sum(w[ind_map[i], i] for i in new_gt) / max(sum(w[:, i].sum() for i in new_gt), 1)
    return float(total), float(old), float(new)

def pair_f1(y, p):
    M = contingency_matrix(y, p, sparse=True)
    c2 = lambda x: float((x * (x - 1) / 2).sum())
    tp = c2(M.data.astype(np.float64))
    pp, tt = c2(np.asarray(M.sum(0)).ravel().astype(np.float64)), c2(np.asarray(M.sum(1)).ravel().astype(np.float64))
    pr, rc = (tp / pp if pp else 0.0), (tp / tt if tt else 0.0)
    return (2 * pr * rc / (pr + rc) if pr + rc else 0.0)

@torch.no_grad()
def predict(model, ds, num_workers=NUM_WORKERS, want_feats=False):
    model.eval(); preds, feats = [], []
    for imgs, _, _, _ in make_test_loader(ds, num_workers):
        imgs = imgs.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(FP16 and DEVICE == "cuda")):
            z = model[0](imgs)
            _, logits = model[1](z)
        preds.append(logits.argmax(1).cpu().numpy())
        if want_feats:
            feats.append(torch.nn.functional.normalize(z.float(), dim=-1).cpu().numpy())
    return np.concatenate(preds), (np.concatenate(feats) if want_feats else None)

def evaluate(model, ds=None, cat=None, roles=None, num_workers=NUM_WORKERS):
    ds = UNL_TEST if ds is None else ds
    cat = UNL_CAT if cat is None else cat
    roles = UNL_ROLE if roles is None else roles
    pred, _ = predict(model, ds, num_workers)
    y = np.array([CLASS_OF[c] for c in cat])
    out = dict(zip(["ACC_all", "ACC_old", "ACC_new"], split_cluster_acc_v2(y, pred, y < N_OLD)))
    for role in ["novel_known_branch", "novel_unseen_branch"]:
        m = roles == role
        if m.sum() > 1:
            out[role] = {"NMI": NMI(y[m], pred[m]), "ARI": ARI(y[m], pred[m]), "pairF1": pair_f1(y[m], pred[m]),
                         "NMI_singletons": NMI(y[m], np.arange(m.sum())), "n_pred_clusters": int(len(np.unique(pred[m])))}
    return out

def show(res, tag=""):
    s = f"{tag} ACC all {res['ACC_all']:.4f} old {res['ACC_old']:.4f} new {res['ACC_new']:.4f}"
    for role in ["novel_known_branch", "novel_unseen_branch"]:
        if role in res:
            r = res[role]
            s += (f" | {role}: NMI {r['NMI']:.3f} (singletons {r['NMI_singletons']:.3f}) "
                  f"ARI {r['ARI']:.3f} pF1 {r['pairF1']:.3f} clusters {r['n_pred_clusters']}")
    print(s, flush=True)
print("evaluation ready")

## Step 7 — Training loop with resume

The loss is the repo's `train.py`, line for line. A checkpoint (`latest.pt`) is written atomically at the end of
**every** epoch, so an interruption loses at most the epoch in progress. The sampler is reseeded with
`SEED + epoch` each epoch.

In [ ]:
# CELL: train_defs
CKPT, LOG = OUT_DIR / "latest.pt", OUT_DIR / "log.json"

def simgcd_step(student, batch, epoch, cluster_criterion):
    images, class_labels, _, mask_lab = batch
    class_labels = class_labels.to(DEVICE, non_blocking=True)
    mask_lab = mask_lab.to(DEVICE, non_blocking=True).bool()
    images = torch.cat(images, dim=0).to(DEVICE, non_blocking=True)
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(FP16 and DEVICE == "cuda")):
        student_proj, student_out = student(images)
        teacher_out = student_out.detach()
        # clustering, sup
        sup_logits = torch.cat([f[mask_lab] for f in (student_out / 0.1).chunk(2)], dim=0)
        sup_labels = torch.cat([class_labels[mask_lab] for _ in range(2)], dim=0)
        cls_loss = nn.CrossEntropyLoss()(sup_logits, sup_labels)
        # clustering, unsup
        cluster_loss = cluster_criterion(student_out, teacher_out, epoch)
        avg_probs = (student_out / 0.1).softmax(dim=1).mean(dim=0)
        me_max_loss = - torch.sum(torch.log(avg_probs**(-avg_probs))) + math.log(float(len(avg_probs)))
        cluster_loss += MEMAX_WEIGHT * me_max_loss
        # representation learning, unsup
        contrastive_logits, contrastive_labels = info_nce_logits(features=student_proj, device=DEVICE)
        contrastive_loss = torch.nn.CrossEntropyLoss()(contrastive_logits, contrastive_labels)
        # representation learning, sup
        student_proj = torch.cat([f[mask_lab].unsqueeze(1) for f in student_proj.chunk(2)], dim=1)
        student_proj = torch.nn.functional.normalize(student_proj, dim=-1)
        sup_con_loss = SupConLoss()(student_proj, labels=class_labels[mask_lab])
        loss = (1 - SUP_WEIGHT) * cluster_loss + SUP_WEIGHT * cls_loss
        loss = loss + (1 - SUP_WEIGHT) * contrastive_loss + SUP_WEIGHT * sup_con_loss
    parts = {"cls": cls_loss.item(), "cluster": cluster_loss.item(), "supcon": sup_con_loss.item(),
             "contrastive": contrastive_loss.item()}
    return loss, parts

def make_optim(student):
    opt = torch.optim.SGD(get_params_groups(student), lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=LR * 1e-3)
    scaler = torch.amp.GradScaler("cuda", enabled=(FP16 and DEVICE == "cuda"))
    crit = DistillLoss(WARMUP_TEACHER_TEMP_EPOCHS, EPOCHS, N_VIEWS, WARMUP_TEACHER_TEMP, TEACHER_TEMP)
    return opt, sched, scaler, crit

def save_ckpt(student, opt, sched, scaler, epoch, hist):
    tmp = CKPT.with_suffix(".tmp")
    torch.save({"epoch": epoch, "model": student.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                "scaler": scaler.state_dict(), "epochs_total": EPOCHS, "torch_rng": torch.get_rng_state(),
                "np_rng": np.random.get_state()}, tmp)
    os.replace(tmp, CKPT)
    json.dump(hist, open(LOG, "w"), indent=1)

def train_simgcd(max_iters=None, num_workers=NUM_WORKERS, resume=True):
    student = build_model()
    opt, sched, scaler, crit = make_optim(student)
    start, hist = 0, []
    if resume and CKPT.exists():
        ck = torch.load(CKPT, map_location=DEVICE, weights_only=False)
        assert ck["epochs_total"] == EPOCHS, f"checkpoint was made with EPOCHS={ck['epochs_total']}; set EPOCHS back"
        student.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
        torch.set_rng_state(ck["torch_rng"].cpu()); np.random.set_state(ck["np_rng"])
        start = ck["epoch"] + 1
        hist = json.load(open(LOG)) if LOG.exists() else []
        print(f"resuming after epoch {start} of {EPOCHS}")
    loader, sampler = make_train_loader(num_workers)
    n_it = len(loader)
    for epoch in range(start, EPOCHS):
        student.train(); sampler.generator.manual_seed(SEED + epoch)
        t0, tot, seen = time.time(), 0.0, 0
        for it, batch in enumerate(loader):
            loss, parts = simgcd_step(student, batch, epoch, crit)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item(); seen += 1
            if it % PRINT_EVERY == 0 or (max_iters and it + 1 == max_iters):
                rate = (it + 1) * BATCH_SIZE / (time.time() - t0)
                print(f"ep {epoch+1}/{EPOCHS} it {it+1}/{n_it} loss {loss.item():.4f} "
                      + " ".join(f"{k} {v:.3f}" for k, v in parts.items())
                      + f" | {rate:.0f} img/s, epoch ~{n_it*BATCH_SIZE/rate/60:.0f} min", flush=True)
            if max_iters and it + 1 >= max_iters:
                return student, {"img_per_s": seen * BATCH_SIZE / (time.time() - t0), "loss": tot / seen}
        sched.step()
        rec = {"epoch": epoch + 1, "loss": tot / max(seen, 1), "lr": opt.param_groups[0]["lr"],
               "minutes": (time.time() - t0) / 60}
        if (epoch + 1) % EVAL_EVERY == 0 or epoch + 1 == EPOCHS:
            rec["eval"] = evaluate(student, num_workers=num_workers); show(rec["eval"], f"[epoch {epoch+1}]")
        hist.append(rec)
        save_ckpt(student, opt, sched, scaler, epoch, hist)
        print(f"epoch {epoch+1} done in {rec['minutes']:.1f} min, mean loss {rec['loss']:.4f}, checkpoint saved", flush=True)
    return student, hist
print("training loop ready")

## Step 8 — Smoke test (about 5 minutes)

Runs 30 training iterations on a fresh model (nothing is saved), then evaluates on 2,000 unlabelled images.
It checks the workers, memory and loss, and projects the time for the full run. **Check the projection
before Step 9.**

In [ ]:
# CELL: smoke
SMOKE_ITERS = 30
try:
    _s, info = train_simgcd(max_iters=SMOKE_ITERS, resume=False)
except Exception as e:
    print("ERROR:", type(e).__name__, e)
    print("If this is a worker/pickling error, set NUM_WORKERS = 0 in Step 1 and rerun from Step 1.")
    print("If it is CUDA out of memory, set BATCH_SIZE = 64 in Step 1 (and say so in the paper).")
    raise
n_it = len(TRAIN_SET) // BATCH_SIZE
ep_min = n_it * BATCH_SIZE / info["img_per_s"] / 60
print(f"\nthroughput {info['img_per_s']:.0f} img/s (2 views each) | one epoch ~{ep_min:.0f} min "
      f"| EPOCHS={EPOCHS}: ~{ep_min*EPOCHS/60:.0f} h of training (+ evaluation every {EVAL_EVERY} epochs)")
for E in (50, 100, 200):
    print(f"   {E:>3} epochs ~ {ep_min*E/60:.0f} h")
if DEVICE == "cuda":
    print(f"peak GPU memory {torch.cuda.max_memory_allocated()/1e9:.1f} GB")
sub = np.random.default_rng(SEED).choice(len(UNL), min(2000, len(UNL)), replace=False)
small = hd.Items(IMG_ROOT, [UNL[i][0] for i in sub], [CLASS_OF[UNL[i][1]] for i in sub], test_tf)
t0 = time.time()
show(evaluate(_s, small, UNL_CAT[sub], UNL_ROLE[sub]), "[smoke eval, 2,000 images]")
print(f"evaluation speed {len(sub)/(time.time()-t0):.0f} img/s")
del _s
if DEVICE == "cuda":
    torch.cuda.empty_cache()

## Step 9 — Full training (resumable)

Safe to interrupt. To continue: rerun Steps 1–7, then this cell. Do not change `EPOCHS` or any hyperparameter
between runs; the cell refuses a checkpoint made with a different `EPOCHS`.

In [ ]:
# CELL: train_full
STUDENT, HIST = train_simgcd()

## Step 10 — Final evaluation and feature export

Uses the final model. Exports, for every image of the 1,817 benchmark species, the L2-normalised DINO CLS feature
after SimGCD fine-tuning (768-d) and the predicted head index, in the same format as the other feature files
(`X`, `cat_ids`, `relpaths`), so the analyses can use SimGCD as a representation.

In [ ]:
# CELL: final
if "STUDENT" not in dir():
    STUDENT = build_model()
    ck = torch.load(CKPT, map_location=DEVICE, weights_only=False)
    assert ck["epoch"] + 1 == EPOCHS, f"training is not finished (last epoch {ck['epoch']+1} of {EPOCHS})"
    STUDENT.load_state_dict(ck["model"])
FINAL = evaluate(STUDENT)
show(FINAL, "[final]")

ALL = [[r, c] for r, c in ITEMS if c in CLASS_OF]
all_ds = hd.Items(IMG_ROOT, [r for r, _ in ALL], [CLASS_OF[c] for _, c in ALL], test_tf)
pred_all, feats = predict(STUDENT, all_ds, want_feats=True)
np.savez(OUT_DIR / "features_simgcd_v2.npz", X=feats.astype(np.float32),
         cat_ids=np.array([c for _, c in ALL], dtype=np.int64), relpaths=np.array([r for r, _ in ALL]),
         pred_head=pred_all.astype(np.int64))
json.dump({"final": FINAL, "epochs": EPOCHS, "batch_size": BATCH_SIZE, "memax_weight": MEMAX_WEIGHT,
           "lr": LR, "weight_decay": WEIGHT_DECAY, "sup_weight": SUP_WEIGHT, "seed": SEED,
           "n_old": N_OLD, "n_classes": N_CLASSES, "labelled": len(LAB), "unlabelled": len(UNL)},
          open(OUT_DIR / "simgcd_results.json", "w"), indent=1, default=float)
print("saved features_simgcd_v2.npz", feats.shape, "and simgcd_results.json")

## Step 11 — What to copy to the analysis machine

In [ ]:
# CELL: manifest
want = ["simgcd_results.json", "log.json", "simgcd_split.json", "features_simgcd_v2.npz"]
total = 0
for f in want:
    p = OUT_DIR / f
    if p.exists():
        total += p.stat().st_size; print(f"  {f:<28}{p.stat().st_size/1e6:>9.1f} MB")
    else:
        print(f"  {f:<28}  MISSING")
print(f"copy these ({total/1e9:.2f} GB) from {OUT_DIR} to the analysis machine. latest.pt can stay on the GPU machine.")